# LAB 01 — Experiments

Notebook gồm:
- Part D — Sparse Representation
- Part F — Preprocessing Ablation
- Part G — Document Search
- Part H — Evaluation
- Part I — Error Analysis
- Part J — Limitation & Hypothesis


In [18]:
!pip -q install transformers

import gzip
import json
import re
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer


In [19]:
path = "/content/c4-train.00000-of-01024-30K.json.gz"

data = []
with gzip.open(path, "rt", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)
docs = df["text"].fillna("").tolist()

print("Number of documents:", len(docs))
print(docs[0][:300])


Number of documents: 30000
Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class fo


## Part D — Experiment 1: Inspect the Sparse Representation

In [20]:
split_data = df["text"].apply(
    lambda x: [w for w in re.split(r"[.,\s]+", x) if w]
)

vectorizer_d = CountVectorizer(analyzer=lambda x: x)
count_matrix = vectorizer_d.fit_transform(split_data)

doc_lengths = np.asarray(count_matrix.sum(axis=1)).ravel()
tf_matrix = count_matrix.multiply(1 / doc_lengths[:, None])

df_vec = count_matrix.getnnz(axis=0)
N = count_matrix.shape[0]
idf_vec = np.log(N / df_vec)

tfidf_matrix = tf_matrix.multiply(idf_vec).tocsr()

V = count_matrix.shape[1]
sparsity = 1 - count_matrix.nnz / (N * V)

print("Number of documents =", N)
print("Vocabulary size =", V)
print("Matrix shape =", tfidf_matrix.shape)
print("Sparsity =", sparsity)


Number of documents = 30000
Vocabulary size = 409872
Matrix shape = (30000, 409872)
Sparsity = 0.9995550261219763


In [21]:
terms = np.array(vectorizer_d.get_feature_names_out())

top_df_idx = np.argsort(df_vec)[::-1][:20]
top_idf_idx = np.argsort(idf_vec)[::-1][:20]

doc_id = 0
doc_vector = tfidf_matrix[doc_id]
indices = doc_vector.indices
values = doc_vector.data
order = np.argsort(values)[::-1][:20]
top_doc_indices = indices[order]
top_doc_values = values[order]

top_df = pd.DataFrame({
    "term": terms[top_df_idx],
    "df": df_vec[top_df_idx]
})

top_idf = pd.DataFrame({
    "term": terms[top_idf_idx],
    "df": df_vec[top_idf_idx],
    "idf": idf_vec[top_idf_idx]
})

top_tfidf = pd.DataFrame({
    "term": terms[top_doc_indices],
    "tfidf": top_doc_values
})

print("20 terms phổ biến nhất theo document frequency")
display(top_df)

print("20 terms có IDF cao nhất")
display(top_idf)

print("20 terms có TF-IDF cao nhất trong document 0")
display(top_tfidf)


20 terms phổ biến nhất theo document frequency


,term,df
0,the,27348
1,and,27270
2,to,26503
3,of,25917
4,a,25436
5,in,24666
6,for,23142
7,is,22570
8,with,20964
9,on,19717


20 terms có IDF cao nhất


,term,df,idf
0,While,1,10.308953
1,Ever,1,10.308953
2,000,1,10.308953
3,,1,10.308953
4,=,1,10.308953
5,,1,10.308953
6,remarkable,1,10.308953
7,!!!!!!,1,10.308953
8,!!!!Some,1,10.308953
9,!!!),1,10.308953


20 terms có TF-IDF cao nhất trong document 0


,term,tfidf
0,BBQ,0.140868
1,Balay,0.079300
2,Missoula!,0.079300
3,BBQ?,0.079300
4,Class,0.076557
5,meat,0.074671
6,KCBS,0.073968
7,Lonestar,0.070849
8,Beginners,0.060854
9,smoker,0.059569


### 7.5 — So sánh và trả lời

- Danh sách **DF cao** thường chứa các từ rất phổ biến trong corpus như `the`, `and`, `to`.
- Danh sách **IDF cao** gồm các term rất hiếm, thường chỉ xuất hiện trong rất ít document; một số term có thể là ký hiệu hoặc token nhiễu.
- Danh sách **TF-IDF cao của một document** thường gồm các từ đặc trưng cho nội dung document đó.

**Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?**  
Không. Nếu term xuất hiện ở nhiều document thì IDF thấp, nên TF-IDF có thể thấp dù term xuất hiện thường xuyên.

**Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?**  
Không. Nếu term không xuất hiện trong một document thì TF = 0, nên TF-IDF của term đó trong document vẫn bằng 0.


## Part F — Experiment 2: Preprocessing Ablation

In [22]:
df["text_lower"] = df["text"].str.lower()
df["token_pa"] = df["text_lower"].apply(lambda x: x.split(" "))

def normalize_punctuation(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r",{2,}", ",", text)
    text = re.sub(r"!+", "!", text)
    text = re.sub(r"\?+", "?", text)
    return text.strip()

df["text_punctuation_normalization"] = df["text_lower"].apply(normalize_punctuation)
tokenizer_pb = df["text_punctuation_normalization"].apply(lambda x: x.split(" "))
df["token_pb"] = tokenizer_pb.apply(
    lambda x: [word for word in x if word not in ENGLISH_STOP_WORDS]
)

subword_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
df["token_pc"] = df["text"].apply(subword_tokenizer.tokenize)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


In [23]:
RELEVANCE = {
    "natural language processing": {
        5912, 8270, 18339, 21879, 23722, 24456
    },
    "deep learning healthcare": {
        8270
    },
    "medical devices pharmacy": {
        12658, 28742
    },
    "healthcare big data": {
        11119
    },
    "GraphQL open source foundation": {
        12574
    }
}

EVAL_QUERIES = list(RELEVANCE.keys())

DEMO_QUERIES = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]


In [24]:
def tok_a(text):
    return text.lower().split(" ")

def tok_b(text):
    text = normalize_punctuation(text)
    return [w for w in text.split(" ") if w not in ENGLISH_STOP_WORDS]

def tok_c(text):
    return subword_tokenizer.tokenize(text)

def build_tfidf(token_series):
    vec = TfidfVectorizer(
        analyzer=lambda x: x,
        lowercase=False
    )
    matrix = vec.fit_transform(token_series)
    return vec, matrix

vec_a, Xa = build_tfidf(df["token_pa"])
vec_b, Xb = build_tfidf(df["token_pb"])
vec_c, Xc = build_tfidf(df["token_pc"])

PIPELINES = {
    "A": (vec_a, Xa, tok_a),
    "B": (vec_b, Xb, tok_b),
    "C": (vec_c, Xc, tok_c)
}


In [25]:
def vocab_size(token_series):
    vocab = set()
    for tokens in token_series:
        vocab.update(tokens)
    return len(vocab)

def avg_tokens(token_series):
    return token_series.apply(len).mean()

def sparsity(matrix):
    return 1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])

def oov_rate(queries, tokenizer_func, vocabulary):
    total = 0
    oov = 0
    for query in queries:
        tokens = tokenizer_func(query)
        total += len(tokens)
        oov += sum(t not in vocabulary for t in tokens)
    return oov / total if total else 0.0

raw_tokens = df["text"].apply(lambda x: x.split(" "))
raw_vocab = vocab_size(raw_tokens)

summary = pd.DataFrame([
    {
        "Pipeline": "A",
        "Vocabulary size": vocab_size(df["token_pa"]),
        "Average tokens/document": avg_tokens(df["token_pa"]),
        "Matrix sparsity": sparsity(Xa),
        "OOV rate": oov_rate(EVAL_QUERIES, tok_a, vec_a.vocabulary_)
    },
    {
        "Pipeline": "B",
        "Vocabulary size": vocab_size(df["token_pb"]),
        "Average tokens/document": avg_tokens(df["token_pb"]),
        "Matrix sparsity": sparsity(Xb),
        "OOV rate": oov_rate(EVAL_QUERIES, tok_b, vec_b.vocabulary_)
    },
    {
        "Pipeline": "C",
        "Vocabulary size": vocab_size(df["token_pc"]),
        "Average tokens/document": avg_tokens(df["token_pc"]),
        "Matrix sparsity": sparsity(Xc),
        "OOV rate": oov_rate(EVAL_QUERIES, tok_c, vec_c.vocabulary_)
    }
])

print("Raw vocabulary before lowercasing:", raw_vocab)
display(summary)


Raw vocabulary before lowercasing: 692173


,Pipeline,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate
0,A,627045,353.607167,0.999709,0.0
1,B,469819,198.020200,0.999699,0.0
2,C,28339,465.099200,0.993200,0.0


Lowercasing làm giảm vocabulary bằng cách gộp các từ chỉ khác nhau ở chữ hoa và chữ thường, từ 692,173 xuống 627,045 từ.

Stopword removal không phải lúc nào cũng tốt hơn; Pipeline B có MRR cao hơn Pipeline A.

Loại punctuation có thể làm mất thông tin về ranh giới câu, viết tắt, URL hoặc nhấn mạnh.

Pipeline A tạo sparse matrix nhất.

Pipeline B cho search tốt nhất với MRR = 0.4468.

Vocabulary nhỏ hơn không đồng nghĩa search tốt hơn vì Pipeline C có vocabulary nhỏ nhất nhưng Pipeline B có MRR cao nhất.


## Part G — Application: Document Search

In [26]:
def search(query, vectorizer, matrix, tokenizer_func, top_k=5):
    query_tokens = tokenizer_func(query)
    q = vectorizer.transform([query_tokens])
    scores = cosine_similarity(q, matrix).ravel()
    ids = np.argsort(scores)[::-1][:top_k]

    return pd.DataFrame([
        {
            "rank": rank,
            "doc_id": int(doc_id),
            "similarity": float(scores[doc_id]),
            "document_preview": docs[doc_id][:200].replace("\n", " ")
        }
        for rank, doc_id in enumerate(ids, start=1)
    ])

for query in DEMO_QUERIES:
    print("\nQUERY:", query)
    display(search(query, vec_b, Xb, tok_b, 5))



QUERY: medical image classification


,rank,doc_id,similarity,document_preview
0,1,18971,0.410989,The new RTS Environmental Classification syste...
1,2,190,0.245536,This title is a comprehensive account of the k...
2,3,12658,0.243912,"This guidance is for pharmacists who handle, u..."
3,4,27781,0.241603,❶Press Officer Resume Sample. Based on your re...
4,5,8370,0.241292,What is a Online Medical Second Opinion? For o...



QUERY: transformer language model


,rank,doc_id,similarity,document_preview
0,1,27936,0.295557,"hi, I am having problems with transformer / ci..."
1,2,24482,0.253559,"Harald, you are a co-owner of Language Partner..."
2,3,25428,0.246510,"Note: If you're on an iPhone, you cannot chang..."
3,4,701,0.232076,Program in Teaching French as a Foreign Langua...
4,5,4075,0.221410,Looking for Spanish language instructor to imp...



QUERY: deep learning healthcare


,rank,doc_id,similarity,document_preview
0,1,11979,0.308109,Doctorate of Healthcare Organization Program i...
1,2,9252,0.295492,"SAN DIEGO AND WASHINGTON, D.C. – Sept. 5, 2018..."
2,3,11119,0.275301,The opportunities offered by Big Data will onl...
3,4,3370,0.237833,These two healthcare REITs are trading for dir...
4,5,15168,0.226812,AI and deep learning is serious business at NV...



QUERY: natural language processing


,rank,doc_id,similarity,document_preview
0,1,8705,0.377575,These regulations may be called the Food Safet...
1,2,24482,0.334029,"Harald, you are a co-owner of Language Partner..."
2,3,25428,0.324743,"Note: If you're on an iPhone, you cannot chang..."
3,4,701,0.305728,Program in Teaching French as a Foreign Langua...
4,5,4075,0.291677,Looking for Spanish language instructor to imp...


## Part H — Evaluation

In [27]:
def precision_at_k(retrieved, relevant, k=5):
    return len(set(retrieved[:k]) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k=5):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)

def reciprocal_rank(ranking, relevant):
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1 / rank
    return 0.0

def evaluate_pipeline(name, vectorizer, matrix, tokenizer_func):
    rows = []

    for query, relevant in RELEVANCE.items():
        q = vectorizer.transform([tokenizer_func(query)])
        scores = cosine_similarity(q, matrix).ravel()
        ranking = np.argsort(scores)[::-1].tolist()
        top5 = ranking[:5]

        rows.append({
            "pipeline": name,
            "query": query,
            "P@5": precision_at_k(top5, relevant, 5),
            "R@5": recall_at_k(top5, relevant, 5),
            "RR": reciprocal_rank(ranking, relevant)
        })

    return pd.DataFrame(rows)

evaluation = pd.concat([
    evaluate_pipeline(name, vec, matrix, tok)
    for name, (vec, matrix, tok) in PIPELINES.items()
], ignore_index=True)

performance = (
    evaluation.groupby("pipeline")[["P@5", "R@5", "RR"]]
    .mean()
    .rename(columns={"RR": "MRR"})
)

display(evaluation)
display(performance)


,pipeline,query,P@5,R@5,RR
0,A,natural language processing,0.0,0.0,0.142857
1,A,deep learning healthcare,0.0,0.0,0.017544
2,A,medical devices pharmacy,0.4,1.0,1.000000
3,A,healthcare big data,0.2,1.0,1.000000
4,A,GraphQL open source foundation,0.0,0.0,0.024390
5,B,natural language processing,0.0,0.0,0.166667
6,B,deep learning healthcare,0.0,0.0,0.017544
7,B,medical devices pharmacy,0.4,1.0,1.000000
8,B,healthcare big data,0.2,1.0,1.000000
9,B,GraphQL open source foundation,0.0,0.0,0.050000


,P@5,R@5,MRR
pipeline,,,
A,0.12,0.4,0.436958
B,0.12,0.4,0.446842
C,0.08,0.3,0.333874


## Part I — Error Analysis

In [28]:
def term_contributions(query, doc_id, vectorizer, matrix, tokenizer_func, top_n=10):
    q = vectorizer.transform([tokenizer_func(query)])
    d = matrix[doc_id]
    product = q.multiply(d).toarray().ravel()
    idx = np.argsort(product)[::-1]
    terms = vectorizer.get_feature_names_out()

    return [
        (terms[i], float(product[i]))
        for i in idx
        if product[i] > 0
    ][:top_n]

def relevant_ranks(query, relevant, vectorizer, matrix, tokenizer_func):
    q = vectorizer.transform([tokenizer_func(query)])
    scores = cosine_similarity(q, matrix).ravel()
    ranking = np.argsort(scores)[::-1]
    rank_of = np.empty(len(ranking), dtype=int)
    rank_of[ranking] = np.arange(1, len(ranking) + 1)

    return {doc_id: int(rank_of[doc_id]) for doc_id in relevant}

def analyse_query(query, relevant, pipeline_name):
    vectorizer, matrix, tokenizer_func = PIPELINES[pipeline_name]
    result = search(query, vectorizer, matrix, tokenizer_func, 5)

    retrieved = result["doc_id"].tolist()
    missed = set(relevant) - set(retrieved)
    top_doc = retrieved[0]

    query_tokens = set(tokenizer_func(query))
    top_tokens = set(tokenizer_func(docs[top_doc]))
    overlap = sorted(query_tokens & top_tokens)

    print("\nQuery:", query)
    print("Expected relevant:", sorted(relevant))
    print("Relevant ranks:", relevant_ranks(query, relevant, vectorizer, matrix, tokenizer_func))
    print("Retrieved top-5:", retrieved)
    print("Missed relevant:", sorted(missed))
    print("Lexical overlap với document đứng đầu:", overlap)
    print("Terms đóng góp similarity:",
          term_contributions(query, top_doc, vectorizer, matrix, tokenizer_func))

    if missed:
        missed_doc = min(
            missed,
            key=lambda d: relevant_ranks(query, relevant, vectorizer, matrix, tokenizer_func)[d]
        )
        missed_overlap = sorted(query_tokens & set(tokenizer_func(docs[missed_doc])))
        print("Lexical overlap với relevant document bị bỏ sót:", missed_overlap)

    display(result)


In [29]:
best_pipeline = performance["MRR"].idxmax()

per_query = evaluation[evaluation["pipeline"] == best_pipeline].copy()
per_query["score"] = per_query["P@5"] + per_query["R@5"] + per_query["RR"]

good_queries = per_query.nlargest(2, "score")["query"].tolist()
bad_queries = per_query.nsmallest(2, "score")["query"].tolist()

print("Pipeline dùng cho error analysis:", best_pipeline)
print("2 queries tốt:", good_queries)
print("2 queries kém:", bad_queries)

for query in good_queries + bad_queries:
    analyse_query(query, RELEVANCE[query], best_pipeline)


Pipeline dùng cho error analysis: B
2 queries tốt: ['medical devices pharmacy', 'healthcare big data']
2 queries kém: ['deep learning healthcare', 'GraphQL open source foundation']

Query: medical devices pharmacy
Expected relevant: [12658, 28742]
Relevant ranks: {12658: 1, 28742: 4}
Retrieved top-5: [12658, 1845, 1078, 28742, 1815]
Missed relevant: []
Lexical overlap với document đứng đầu: ['devices', 'medical', 'pharmacy']
Terms đóng góp similarity: [('medical', 0.2345482043217411), ('devices', 0.16577491795910057), ('pharmacy', 0.09895797315636835)]


,rank,doc_id,similarity,document_preview
0,1,12658,0.499281,"This guidance is for pharmacists who handle, u..."
1,2,1845,0.397088,Whoa!! The Take care Pharmacy Mobile applicati...
2,3,1078,0.385078,Pharmacy job opportunities at Paydens Limited....
3,4,28742,0.366787,Northern Pharmacy & Medical Equipment has a fu...
4,5,1815,0.341186,U.S. Pharmacist is a monthly journal dedicated...



Query: healthcare big data
Expected relevant: [11119]
Relevant ranks: {11119: 1}
Retrieved top-5: [11119, 4770, 2153, 1779, 11979]
Missed relevant: []
Lexical overlap với document đứng đầu: ['big', 'data', 'healthcare']
Terms đóng góp similarity: [('healthcare', 0.30194490663954276), ('data', 0.1979884577353296), ('big', 0.10132903865030274)]


,rank,doc_id,similarity,document_preview
0,1,11119,0.601262,The opportunities offered by Big Data will onl...
1,2,4770,0.364365,One of the many changes that the new Regulatio...
2,3,2153,0.349305,The democratization of data is a real phenomen...
3,4,1779,0.338565,How do companies innovate with big data? It st...
4,5,11979,0.337929,Doctorate of Healthcare Organization Program i...



Query: deep learning healthcare
Expected relevant: [8270]
Relevant ranks: {8270: 57}
Retrieved top-5: [11979, 9252, 11119, 3370, 15168]
Missed relevant: [8270]
Lexical overlap với document đứng đầu: ['healthcare']
Terms đóng góp similarity: [('healthcare', 0.30810916619557727)]
Lexical overlap với relevant document bị bỏ sót: ['deep', 'healthcare', 'learning']


,rank,doc_id,similarity,document_preview
0,1,11979,0.308109,Doctorate of Healthcare Organization Program i...
1,2,9252,0.295492,"SAN DIEGO AND WASHINGTON, D.C. – Sept. 5, 2018..."
2,3,11119,0.275301,The opportunities offered by Big Data will onl...
3,4,3370,0.237833,These two healthcare REITs are trading for dir...
4,5,15168,0.226812,AI and deep learning is serious business at NV...



Query: GraphQL open source foundation
Expected relevant: [12574]
Relevant ranks: {12574: 20}
Retrieved top-5: [1963, 19471, 10006, 11930, 25092]
Missed relevant: [12574]
Lexical overlap với document đứng đầu: ['open', 'source']
Terms đóng góp similarity: [('source', 0.11181051047699489), ('open', 0.06415448928045357)]
Lexical overlap với relevant document bị bỏ sót: ['foundation', 'graphql', 'open']


,rank,doc_id,similarity,document_preview
0,1,1963,0.175965,What does it take to get a start-up going? Wel...
1,2,19471,0.167681,"Of course the open source scene is changing, b..."
2,3,10006,0.159321,This KB article provides a better understandin...
3,4,11930,0.156605,The W3C moved beyond its initial Web of Things...
4,5,25092,0.154208,This is the second half of a two part article....


##Analysis




1. Document đứng đầu vì có cosine similarity cao nhất với query.
2. Các term đóng góp nhiều nhất được in trong Terms đóng góp similarity.
3. Lexical overlap cho biết query và document có chia sẻ token hay không.
4. Missed relevant cho biết relevant document nào không nằm trong top-5.
5. Nếu relevant document gần nghĩa nhưng có ít hoặc không có lexical overlap, failure chủ yếu đến từ lexical matching. Nếu có overlap nhưng rank vẫn thấp, cần xem thêm TF-IDF weighting, preprocessing và vocabulary.


### Failure case quan trọng nhất

In [30]:
failures = []

vec, matrix, tok = PIPELINES[best_pipeline]

for query, relevant in RELEVANCE.items():
    ranks = relevant_ranks(query, relevant, vec, matrix, tok)
    for doc_id, rank in ranks.items():
        if rank > 5:
            failures.append((rank, query, doc_id))

if failures:
    rank, query, doc_id = max(failures)

    query_tokens = set(tok(query))
    doc_tokens = set(tok(docs[doc_id]))
    overlap = sorted(query_tokens & doc_tokens)

    print("Query:", query)
    print("Relevant document:", doc_id)
    print("Rank:", rank)
    print("Preview:", docs[doc_id][:500].replace("\n", " "))
    print("Lexical overlap:", overlap)

Query: natural language processing
Relevant document: 24456
Rank: 356
Preview: Admit it, sifting through an overwhelming number of CVs is not fun. It’s also often a waste of time considering that most applicants do not fit job requirements. You already have enough on your plate. The lengthy hiring process negatively affects candidates too—if waiting a long time, potential candidates may be deterred from an open position and into that of a competitor’s. AI can help recruiters throughout the whole recruitment process. Well, nowadays, you’re in luck. Artificial intelligence e
Lexical overlap: ['language', 'natural', 'processing']


Giải thích: có lexical overlap nhưng document vẫn xếp thấp.

Nguyên nhân có thể đến từ TF-IDF weighting, preprocessing hoặc nhiều document khác có overlap mạnh hơn.

## Part J — From Failure to the Next NLP Representation

TF-IDF chủ yếu biểu diễn document bằng thống kê từ và lexical overlap. Vì vậy hai biểu thức gần nghĩa nhưng dùng các từ khác nhau có thể có similarity thấp.

**Hypothesis:** dùng **word embedding** hoặc **contextual embedding/Transformer** có thể biểu diễn similarity về nghĩa tốt hơn, vì các từ hoặc câu có ngữ nghĩa gần nhau có thể được đặt gần nhau trong không gian vector dù không dùng cùng token.


## Export `results.csv`

In [31]:
rows = []

for pipeline_name, (vectorizer, matrix, tokenizer_func) in PIPELINES.items():
    metrics = evaluation[evaluation["pipeline"] == pipeline_name].set_index("query")

    for query, relevant in RELEVANCE.items():
        result = search(query, vectorizer, matrix, tokenizer_func, 5)

        for _, r in result.iterrows():
            rows.append({
                "pipeline": pipeline_name,
                "query": query,
                "rank": int(r["rank"]),
                "doc_id": int(r["doc_id"]),
                "similarity": float(r["similarity"]),
                "is_relevant": int(r["doc_id"] in relevant),
                "P@5": metrics.loc[query, "P@5"],
                "R@5": metrics.loc[query, "R@5"],
                "RR": metrics.loc[query, "RR"],
                "document_preview": r["document_preview"]
            })

results = pd.DataFrame(rows)
results.to_csv("results.csv", index=False)

print("Saved results.csv")
display(results.head())


Saved results.csv


,pipeline,query,rank,doc_id,similarity,is_relevant,P@5,R@5,RR,document_preview
0,A,natural language processing,1,8705,0.336476,0,0.0,0.0,0.142857,These regulations may be called the Food Safet...
1,A,natural language processing,2,4075,0.315685,0,0.0,0.0,0.142857,Looking for Spanish language instructor to imp...
2,A,natural language processing,3,25428,0.301040,0,0.0,0.0,0.142857,"Note: If you're on an iPhone, you cannot chang..."
3,A,natural language processing,4,701,0.290095,0,0.0,0.0,0.142857,Program in Teaching French as a Foreign Langua...
4,A,natural language processing,5,24482,0.261873,0,0.0,0.0,0.142857,"Harald, you are a co-owner of Language Partner..."
